
<div style="text-align: center; padding: 20px;">
    <h1 style="font-size: 28px; font-weight: bold;">1ο Εργαστηριακό Project</h1>
    <h2 style="font-size: 20px; font-weight: normal; color: white;">
        Εντοπισμός Σημείων Ενδιαφέροντος και Εξαγωγή Χαρακτηριστικών σε Εικόνες
    </h2>
</div>

Απαιτούμενες βιβλιοθήκες:

In [ ]:
from cv26_lab1_part2_utils import interest_points_visualization, disk_strel
import numpy as np
import cv2
import matplotlib.pyplot as plt
import cv26_lab1_part3_utils as p3
from torchvision.models import mobilenet_v3_small
import os
from PIL import Image
import torch
import torchvision
from torchvision.models import MobileNet_V3_Small_Weights

## Μέρος 1

### 1.1

#### 1.1.1

In [ ]:
# Reading the image I0
I0 = cv2.imread('edgetest_26.png', cv2.IMREAD_GRAYSCALE)

# Displaying the image I0
plt.imshow(I0, cmap='gray')
plt.title('Original Grayscale Image')
print ('Original Image Shape:', I0.shape) # Shape of the original image
plt.axis('off')
plt.show()

#### 1.1.2

In [ ]:
#Adding Gaussian noise to the image I0, with mean 0 and custom PSNR
mean = 0

# Calculated standard deviation for the desired PSNR
desired_std1 = 25.5 # Desired PSNR in 20dB
desired_std2 = 80.638 # for 10dB

noise1 = np.random.normal(mean, desired_std1, I0.shape).astype(np.float64)
noise2 = np.random.normal(mean, desired_std2, I0.shape).astype(np.float64)

# Adding the noise
I1 = np.clip(I0 + noise1, 0, 255).astype(np.uint8)
I2 = np.clip(I0 + noise2, 0, 255).astype(np.uint8)


fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(I1, cmap='gray')
axes[0].set_title('Noisy Image with PSNR ~ 20dB')
axes[0].axis('off')

axes[1].imshow(I2, cmap='gray')
axes[1].set_title('Noisy Image with PSNR ~ 10dB')
axes[1].axis('off')
plt.tight_layout()
plt.show()

cv2.imwrite('images/images_1.1.2/noisy_image_20dB.png', I1)
cv2.imwrite('images/images_1.1.2/noisy_image_10dB.png', I2)

#### 1.2.1

In [ ]:
# Function that makes a custom Gaussian Kernel and a LoG Kernel
def gaussian_kernel_generator(sigma):
    n = int(np.ceil(3 * sigma)*2 + 1)  # Kernel size based on sigma

    gaussian = cv2.getGaussianKernel(n, sigma)
    gaussian_kernel = gaussian @ gaussian.T  # Create 2D Gaussian kernel

    # Create Laplacian of Gaussian (LoG) kernel, using the formula
    x = np.arange(-(n//2), n//2 +1)
    y = np.arange(-(n//2), n//2 +1)
    x, y = np.meshgrid(x, y) # Create a grid of (x, y) coordinates

    dist = x**2 + y**2

    # LoG kernel formula
    log_kernel = (dist - 2 * sigma**2) * np.exp(-dist / (2 * sigma**2)) / (sigma**6 * 2 * np.pi)

    #make average of the kernel equal to 0, λογικά θα είναι ήδη κοντά στο 0, αλλά για να είμαστε σίγουροι
    log_kernel -= log_kernel.mean()

    return gaussian_kernel, log_kernel

#### 1.2.2

In [ ]:
# Function that applies the custom LoG kernel and the non-linear filters to the noisy images I1 and I2
def apply_filters(I, sigma):
    Gsigma, LoGsigma = gaussian_kernel_generator(sigma)

    # Calculate Laplacian of Gaussian (LoG) of the noisy image I using the custom LoG kernel
    LoG_result = cv2.filter2D(I.astype(np.float32), cv2.CV_32F, LoGsigma.astype(np.float32))

    # Calculate Laplacian using non-lineaer filters
    B = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=np.uint8) # Kernel

    Isigma = cv2.filter2D(I.astype(np.float32), cv2.CV_32F, Gsigma.astype(np.float32)) # First apply Gaussian smoothing
    Non_Linear_result = cv2.dilate(Isigma, B) + cv2.erode(Isigma, B) - 2 * Isigma # Laplacian using dilation and erosion

    return LoG_result, Non_Linear_result

In [ ]:
# Test apply_filters
sigma = 2.0
LoG_I1, Non_Linear_I1 = apply_filters(I1, sigma)

#plot the results
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(LoG_I1, cmap='gray')
axes[0].set_title('LoG Filtered Image (with 20dB PSNR)')
axes[0].axis('off') 
axes[1].imshow(Non_Linear_I1, cmap='gray')
axes[1].set_title('Non-Linear Filtered Image (with 20dB PSNR)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

#save the plot, with cmap gray
# Κανονικοποίηση των τιμών στο εύρος 0-255 και μετατροπή σε 8-bit
LoG_I1_saved = cv2.normalize(LoG_I1, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
Non_Linear_I1_saved = cv2.normalize(Non_Linear_I1, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
cv2.imwrite('images/images_1.2.2/LoG_filtered_image_I1.png', LoG_I1_saved)
cv2.imwrite('images/images_1.2.2/Non_Linear_filtered_image_I1.png', Non_Linear_I1_saved)

#### 1.2.3

In [ ]:
# Function that finds the zero-crossings in the LoG result and the non-linear result, and creates binary edge maps
def zero_crossing_edge_map(Laplacian_Result):

    #when L is greateer than 0, we set it to 255, otherwise we set it to 0
    X = (Laplacian_Result >= 0).astype(np.uint8)

    B = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=np.uint8) # Kernel

    Y = cv2.dilate(X, B) - cv2.erode(X, B) # Find the zero-crossings using dilation and erosion

    edge_map = (Y > 0).astype(np.uint8) # Create binary edge map

    return edge_map

#### 1.2.4

In [ ]:
# Make Edge Detection function
def EdgeDetect(I, sigma, thetaedge, type):
    LoG_result, Non_Linear_result = apply_filters(I, sigma)

    gaus, _ = gaussian_kernel_generator(sigma)
    gaus_result = cv2.filter2D(I.astype(np.float32), cv2.CV_32F, gaus.astype(np.float32))

    edge_map = None

    if type == 'linear':
        edge_map = zero_crossing_edge_map(LoG_result)
    else:
        edge_map = zero_crossing_edge_map(Non_Linear_result)
    
    # Apply thresholding

    gy, gx = np.gradient(gaus_result)
    grad_mag = np.sqrt(gx**2 + gy**2)
    threshold = thetaedge * grad_mag.max()  

    edge_map = np.where((edge_map == 1) & (grad_mag > threshold), 1, 0).astype(np.uint8) #είναι πιο γρήγορο με binary AND, αφού το edge_map είναι binary.

    # return the binary edge map
    return edge_map

In [ ]:
# Testing the function on the noisy images I1 and I2 with different sigma and thetaedge values
detected1 = EdgeDetect(I1, sigma=1.5, thetaedge=0.2, type='linear')
detected2 = EdgeDetect(I1, sigma=1.5, thetaedge=0.2, type='non-linear')

#save the edge maps, multiplying by 255 to make them visible
cv2.imwrite('images/images_1.2.4/edge_map_I1_linear.png', detected1 * 255) # Save the edge map for linear method
cv2.imwrite('images/images_1.2.4/edge_map_I1_non_linear.png', detected2 * 255) # Save the edge map for non-linear method

detected3 = EdgeDetect(I2, sigma=3, thetaedge=0.2, type='linear')
detected4 = EdgeDetect(I2, sigma=3, thetaedge=0.2, type='non-linear')

# Displaying the edge maps in a grid
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(detected1, cmap='binary')
axes[0].set_title('Edge Map using LoG (Linear)')
axes[0].axis('off') 
axes[1].imshow(detected2, cmap='binary')
axes[1].set_title('Edge Map using Non-Linear Filters')
axes[1].axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(detected3, cmap='binary')
axes[0].set_title('Edge Map using LoG (Linear)')
axes[0].axis('off')
axes[1].imshow(detected4, cmap='binary')
axes[1].set_title('Edge Map using Non-Linear Filters')
axes[1].axis('off')
plt.tight_layout()
plt.show()

### 1.3

In [ ]:
theta_real_edge = 2.0 # Threshold for edge detection

# Edge operator on original image I0
B = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=np.uint8) # Kernel

M = cv2.dilate(I0, B) - cv2.erode(I0, B) # Edge map using dilation and erosion on the original image I0

T = np.where(M > theta_real_edge, 1, 0).astype(np.uint8) # Binary edge map from the original image I0

# Displaying the edge map from the original image I0
plt.imshow(T, cmap='binary')
plt.title('Edge Map from Original Image I0')
plt.axis('off')
plt.show()

cv2.imwrite('images/images_1.3.1/real_edge_map.png', T*255) # Save the edge map for linear method

In [ ]:
# A function that calculates precision and recall
def calculate_precision_recall(detected_edges, true_edges):
    C_T_D = (detected_edges & true_edges).sum()  # Card of T and D (True Positives)
    C_T = detected_edges.sum() # Card of T (True Positives + False Positives)
    C_D = true_edges.sum()    # Card of D (True Positives + False Negatives)

    precision = C_T_D / C_T if (C_T) > 0 else 0
    recall = C_T_D / C_D if (C_D) > 0 else 0

    average_precision_recall = (precision + recall) / 2

    return precision, recall, average_precision_recall

In [ ]:
# Calculate precision and recall for the edge maps detected from I1 and I2
precision1, recall1, C1 = calculate_precision_recall(detected1, T) # for LoG (Linear) on I1
precision2, recall2, C2 = calculate_precision_recall(detected2, T) # for Non-Linear Filters on I1
precision3, recall3, C3 = calculate_precision_recall(detected3, T) # for LoG (Linear) on I2
precision4, recall4, C4 = calculate_precision_recall(detected4, T) # for Non-Linear Filters on I2

# Displaying the precision and recall values
print(f"LoG (Linear) on I1: Precision = {precision1:.4f}, Recall = {recall1:.4f}, Average = {C1:.4f}")
print(f"Non-Linear Filters on I1: Precision = {precision2:.4f}, Recall = {recall2:.4f}, Average = {C2:.4f}")    
print(f"LoG (Linear) on I2: Precision = {precision3:.4f}, Recall = {recall3:.4f}, Average = {C3:.4f}")
print(f"Non-Linear Filters on I2: Precision = {precision4:.4f}, Recall = {recall4:.4f}, Average = {C4:.4f}")

In [ ]:
# Different values of sigma and thetaedge, and calculate the corresponding C values for both methods, to find the best parameters
sigma_values = np.linspace(0.5, 4.0, 20)
thetaedge_values = np.linspace(0.1, 0.5, 20)
C_values_linear = np.zeros((len(sigma_values), len(thetaedge_values)))
C_values_non_linear = np.zeros((len(sigma_values), len(thetaedge_values)))
for i, sigma in enumerate(sigma_values):
    for j, thetaedge in enumerate(thetaedge_values):
        detected_linear = EdgeDetect(I1, sigma, thetaedge, type='linear')
        detected_non_linear = EdgeDetect(I1, sigma, thetaedge, type='non-linear')

        _, _, C_values_linear[i, j] = calculate_precision_recall(detected_linear, T)
        _, _, C_values_non_linear[i, j] = calculate_precision_recall(detected_non_linear, T)

# Find the best C value
best_C_linear = C_values_linear.max()
best_C_non_linear = C_values_non_linear.max()
best_sigma_linear, best_thetaedge_linear = np.unravel_index(C_values_linear.argmax(), C_values_linear.shape)
best_sigma_non_linear, best_thetaedge_non_linear = np.unravel_index(C_values_non_linear.argmax(), C_values_non_linear.shape)
print(f"Best C for Linear Method: {best_C_linear:.4f} at sigma = {sigma_values[best_sigma_linear]:.2f}, thetaedge = {thetaedge_values[best_thetaedge_linear]:.2f}")
print(f"Best C for Non-Linear Method: {best_C_non_linear:.4f} at sigma = {sigma_values[best_sigma_non_linear]:.2f}, thetaedge = {thetaedge_values[best_thetaedge_non_linear]:.2f}")

#plot the best edge maps for both methods
best_edge_map_linear = EdgeDetect(I1, sigma_values[best_sigma_linear], thetaedge_values[best_thetaedge_linear], type='linear')
best_edge_map_non_linear = EdgeDetect(I1, sigma_values[best_sigma_non_linear], thetaedge_values[best_thetaedge_non_linear], type='non-linear')
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(best_edge_map_linear, cmap='gray')
axes[0].set_title(f'Best Edge Map (Linear) - C={best_C_linear:.4f}')
axes[0].axis('off')
axes[1].imshow(best_edge_map_non_linear, cmap='gray')
axes[1].set_title(f'Best Edge Map (Non-Linear) - C={best_C_non_linear:.4f}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Plot the C values as a function of sigma and thetaedge for both methods, as heatmaps, for both methods, for the noisy image I1
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(C_values_linear, extent=(thetaedge_values.min(), thetaedge_values.max(), sigma_values.min(), sigma_values.max()), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='C Value')
plt.title('C Values for Linear Method (I1)')
plt.xlabel('Thetaedge')
plt.ylabel('Sigma')
plt.subplot(1, 2, 2)
plt.imshow(C_values_non_linear, extent=(thetaedge_values.min(), thetaedge_values.max(), sigma_values.min(), sigma_values.max()), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='C Value')
plt.title('C Values for Non-Linear Method (I1)')
plt.xlabel('Thetaedge')
plt.ylabel('Sigma')
plt.tight_layout()
plt.show()

In [ ]:
# Same proccess for the noisy image I2
sigma_values = np.linspace(0.5, 4.0, 20)
thetaedge_values = np.linspace(0.1, 0.5, 20)
C_values_linear = np.zeros((len(sigma_values), len(thetaedge_values)))
C_values_non_linear = np.zeros((len(sigma_values), len(thetaedge_values)))
for i, sigma in enumerate(sigma_values):
    for j, thetaedge in enumerate(thetaedge_values):
        detected_linear = EdgeDetect(I2, sigma, thetaedge, type='linear')
        detected_non_linear = EdgeDetect(I2, sigma, thetaedge, type='non-linear')

        _, _, C_values_linear[i, j] = calculate_precision_recall(detected_linear, T)
        _, _, C_values_non_linear[i, j] = calculate_precision_recall(detected_non_linear, T)

# Find the best C value
best_C_linear = C_values_linear.max()
best_C_non_linear = C_values_non_linear.max()
best_sigma_linear, best_thetaedge_linear = np.unravel_index(C_values_linear.argmax(), C_values_linear.shape)
best_sigma_non_linear, best_thetaedge_non_linear = np.unravel_index(C_values_non_linear.argmax(), C_values_non_linear.shape)
print(f"Best C for Linear Method: {best_C_linear:.4f} at sigma = {sigma_values[best_sigma_linear]:.2f}, thetaedge = {thetaedge_values[best_thetaedge_linear]:.2f}")
print(f"Best C for Non-Linear Method: {best_C_non_linear:.4f} at sigma = {sigma_values[best_sigma_non_linear]:.2f}, thetaedge = {thetaedge_values[best_thetaedge_non_linear]:.2f}")

#plot the best edge maps for both methods
best_edge_map_linear = EdgeDetect(I2, sigma_values[best_sigma_linear], thetaedge_values[best_thetaedge_linear], type='linear')
best_edge_map_non_linear = EdgeDetect(I2, sigma_values[best_sigma_non_linear], thetaedge_values[best_thetaedge_non_linear], type='non-linear')
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(best_edge_map_linear, cmap='gray')
axes[0].set_title(f'Best Edge Map (Linear) - C={best_C_linear:.4f}')
axes[0].axis('off')
axes[1].imshow(best_edge_map_non_linear, cmap='gray')
axes[1].set_title(f'Best Edge Map (Non-Linear) - C={best_C_non_linear:.4f}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Plot the C values as a function of sigma and thetaedge for both methods, as heatmaps, for both methods, for the noisy image I2
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(C_values_linear, extent=(thetaedge_values.min(), thetaedge_values.max(), sigma_values.min(), sigma_values.max()), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='C Value')
plt.title('C Values for Linear Method (I2)')
plt.xlabel('Thetaedge')
plt.ylabel('Sigma')
plt.subplot(1, 2, 2)
plt.imshow(C_values_non_linear, extent=(thetaedge_values.min(), thetaedge_values.max(), sigma_values.min(), sigma_values.max()), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='C Value')
plt.title('C Values for Non-Linear Method (I2)')
plt.xlabel('Thetaedge')
plt.ylabel('Sigma')
plt.tight_layout()
plt.show()

### 1.4

In [ ]:
image_new  =  cv2.imread('ermoupoli.jpg', cv2.IMREAD_GRAYSCALE)
# Displaying the image I0
plt.imshow(image_new, cmap='gray')
plt.title('Original Grayscale Image')
print ('Original Image Shape:', image_new.shape) # Shape of the original image
plt.axis('off')
plt.show()

In [ ]:
# Find ground truth edges for the new image
theta_real_edge_2 = 90.0 # Threshold for edge detection

# Edge operator on image_new
M2 = cv2.dilate(image_new, B) - cv2.erode(image_new, B) # Edge map using dilation and erosion on the new image

T2 = np.where(M2 > theta_real_edge_2, 1, 0).astype(np.uint8) # Binary edge map from the new image

# Displaying the edge map from the new image
plt.imshow(T2, cmap='gray')
plt.title('Edge Map from New Image')
plt.axis('off')
plt.show()

cv2.imwrite('images/images_1.4.1/real_edge_map2.png', T2*255) # Save the edge map for linear method

In [ ]:
# Find best edge maps for the new image
sigma_values = [0.5, 1.0, 1.5, 2.0]
thetaedge_values = [0.1, 0.15, 0.18, 0.2, 0.25]

edge_maps_non_linear = []
#nested loops to find edge maps

for theta in thetaedge_values:
    for sigma in sigma_values:
        edge_map_non_linear = EdgeDetect(image_new, sigma, theta, type='non-linear')
        edge_maps_non_linear.append((edge_map_non_linear, sigma, theta))

#plot the edge maps for the non-linear method
fig, axes = plt.subplots(len(thetaedge_values), len(sigma_values), figsize=(15, 15))
for i, theta in enumerate(thetaedge_values):
    for j, sigma in enumerate(sigma_values):
        edge_map_non_linear = edge_maps_non_linear[i*len(sigma_values) + j][0]
        axes[i, j].imshow(edge_map_non_linear, cmap='gray')
        axes[i, j].set_title(f'Non-Linear - Sigma: {sigma}, Theta: {theta}')
        axes[i, j].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#plot the sigma=2 and theta=0.2 edge map, which is the best one for the non-linear method, according to the previous results
best_edge_map_non_linear = EdgeDetect(image_new, sigma=2.0, thetaedge=0.25, type='non-linear')
plt.imshow(best_edge_map_non_linear, cmap='gray')
plt.title('Best Edge Map (Non-Linear) - Sigma: 2.0, Theta: 0.2')
plt.axis('off')
plt.show()

## Μέρος 2

### 2.1

In [ ]:
# Function that finds corners in the image using the Harris corner detection method
def find_corners(image, sg, p, k, theta_corner, return_log=False):

    image = image.astype(np.float64) / image.max() # Normalize to [0,1] for numerical stability and consistent thresholding

    # Calculate the size of the Gaussian kernels based on the standard deviations
    Ns = 2 * np.ceil(3 * sg) + 1
    Np = 2 * np.ceil(3 * p) + 1

    # Create the Gaussian kernels for smoothing 
    Gs_1d = cv2.getGaussianKernel(int(Ns), sg)
    Gp_1d = cv2.getGaussianKernel(int(Np), p)

    Gs = Gs_1d @ Gs_1d.T # Symmetric gaussian kernel
    Gp = Gp_1d @ Gp_1d.T # Symmetric gaussian kernel

    # Convolution of the image with the Gaussian kernel
    Is = cv2.filter2D(image, -1, Gs)

    if return_log:
        # 2nd order derivatives of the smoothed image for the Laplacian of Gaussian (LoG) computation
        grad_x = np.gradient(Is, axis=1)
        grad_y = np.gradient(Is, axis=0)
        Isxx = np.gradient(grad_x, axis=1)
        Isyy = np.gradient(grad_y, axis=0)
        LoG = (sg ** 2) * np.abs(Isxx + Isyy)
    else:
        LoG = None

    dIs = np.gradient(Is) # Gradient of the smoothed image

    dIs_x = np.array(dIs[0], dtype=np.float64)  # Gradient in x direction
    dIs_y = np.array(dIs[1], dtype=np.float64)  # Gradient in y direction

    product_x = dIs_x * dIs_x # Product of the gradient in x direction
    product_y = dIs_y * dIs_y # Product of the gradient in y direction
    product_xy = dIs_x * dIs_y # Product of the gradient in xy direction

    # Convolution of the products with the Gaussian kernel
    J1 = cv2.filter2D(product_x, -1, Gp)
    J2 = cv2.filter2D(product_xy, -1, Gp)
    J3 = cv2.filter2D(product_y, -1, Gp)

    l_plus = 1/2 * (J1 + J3 + np.sqrt((J1 - J3) ** 2 + 4 * J2 ** 2)) # Largest eigenvalue
    l_minus = 1/2 * (J1 + J3 - np.sqrt((J1 - J3) ** 2 + 4 * J2 ** 2)) # Smallest eigenvalue

    R = l_plus * l_minus - k * (l_plus + l_minus) ** 2 # Harris response

    B_sq = disk_strel(Ns) # Structuring element for dilation

    local_max = (R == cv2.dilate(R, B_sq)) # Local maxima of the Harris response

    Rmax = np.max(R) # Maximum value of the Harris response
    above_threshold = R > theta_corner * Rmax # Thresholding the Harris response

    corners_mask = local_max & above_threshold # Mask of the corners

    y_corners, x_corners = np.where(corners_mask) # Coordinates of the corners

    corners_list = np.column_stack([x_corners, y_corners, sg*np.ones(len(x_corners))])
    
    if return_log:
        return corners_list, l_plus, l_minus, LoG
    else:
        return corners_list, l_plus, l_minus

#### 2.1.2

In [ ]:
# Read the images
image1 = cv2.imread('solar.jpg')
image2 = cv2.imread('blood_cells.jpg')
image3 = cv2.imread('solar_edge.jpg')

# Convert BGR to RGB for visualization
image1_rgb = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)
image2_rgb = cv2.cvtColor(image2, cv2.COLOR_BGR2RGB)
image3_rgb = cv2.cvtColor(image3, cv2.COLOR_BGR2RGB)

# Convert to grayscale for corner detection
image1_gray = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
image2_gray = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)
image3_gray = cv2.cvtColor(image3, cv2.COLOR_BGR2GRAY)

images = {
    'solar':  (image1_rgb, image1_gray),
    'solar_edge': (image3_rgb, image3_gray),
    'blood_cells': (image2_rgb, image2_gray)
}

Σχεδιάζουμε τις γκρίζες εικόνες $λ_+, λ_-$ :

In [ ]:
# Parameters for corner detection
sg = 2
p = 2.5
k_harris = 0.05
theta_corner = 0.005

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

col_titles = ['Αρχική εικόνα', 'λ+ (μεγαλύτερη ιδιοτιμή)', 'λ- (μικρότερη ιδιοτιμή)']

for col, title in enumerate(col_titles):
    axes[col].set_title(title, fontsize=18, fontweight='medium', pad=15)

name, (img_rgb, img_gray) = list(images.items())[0]

corners_list, l_plus, l_minus = find_corners(img_gray, sg, p, k_harris, theta_corner)

axes[0].imshow(img_rgb)
axes[0].axis('off')

axes[1].imshow(l_plus, cmap='gray')
axes[1].axis('off')

axes[2].imshow(l_minus, cmap='gray')
axes[2].axis('off')

plt.tight_layout()
plt.show()

Αν τώρα αυξήσουμε το σ:

In [ ]:
sg = 4
p = 2.5
k_harris = 0.05
theta_corner = 0.005

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

col_titles = ['Αρχική εικόνα', 'λ+ (μεγαλύτερη ιδιοτιμή)', 'λ- (μικρότερη ιδιοτιμή)']

for col, title in enumerate(col_titles):
    axes[col].set_title(title, fontsize=18, fontweight='medium', pad=15)

name, (img_rgb, img_gray) = list(images.items())[0]

corners_list, l_plus, l_minus = find_corners(img_gray, sg, p, k_harris, theta_corner)

axes[0].imshow(img_rgb)
axes[0].axis('off')

axes[1].imshow(l_plus, cmap='gray')
axes[1].axis('off')

axes[2].imshow(l_minus, cmap='gray')
axes[2].axis('off')

plt.tight_layout()
plt.show()

Αν αυξήσουμε το $p$:

In [ ]:
sg = 2
p = 4
k_harris = 0.05
theta_corner = 0.005

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

col_titles = ['Αρχική εικόνα', 'λ+ (μεγαλύτερη ιδιοτιμή)', 'λ- (μικρότερη ιδιοτιμή)']

for col, title in enumerate(col_titles):
    axes[col].set_title(title, fontsize=18, fontweight='medium', pad=15)

name, (img_rgb, img_gray) = list(images.items())[0]

corners_list, l_plus, l_minus = find_corners(img_gray, sg, p, k_harris, theta_corner)

axes[0].imshow(img_rgb)
axes[0].axis('off')

axes[1].imshow(l_plus, cmap='gray')
axes[1].axis('off')

axes[2].imshow(l_minus, cmap='gray')
axes[2].axis('off')

plt.tight_layout()
plt.show()

#### 2.1.3

Στη συνέχεια, ανιχνεύουμε τις γωνίες της εικόνας solar και αυτής που βγάλαμε ως αποτέλεσμα στο $1^\text{ο}$ μέρος, με είσοδο την ίδια εικόνα.

In [ ]:
sg = 2
p = 2.5
k_harris = 0.05
theta_corner = 0.005

selected = list(images.items())[0:2]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for row, (name, (img_rgb, img_gray)) in enumerate(selected):
    
    corners_list, _, _ = find_corners(img_gray, sg, p, k_harris, theta_corner)

    # Original image visualization
    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title(f"{name} - original", fontsize=14, fontweight='medium', pad=15)
    axes[row, 0].axis('off')

    # Corners visualization
    interest_points_visualization(img_rgb, corners_list, ax=axes[row, 1])
    axes[row, 1].set_title(f"{name} - corners", fontsize=14, fontweight='medium', pad=15)
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()

Για διαφορετικές τιμές του $σ$ έχουμε:

In [ ]:
p = 2.5
k_harris = 0.05
theta_corner = 0.005

sg_values = [0.5, 2, 4.0]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, sg in enumerate(sg_values):
    corners_list, _, _ = find_corners(image1_gray, sg, p, k_harris, theta_corner)
    interest_points_visualization(image1_rgb, corners_list, ax=axes[col])
    axes[col].set_title(f'σ = {sg}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

Για διαφορετικές τιμές του $p$ έχουμε:

In [ ]:
sg = 2
k_harris = 0.05
theta_corner = 0.005

p_values = [0.5, 2.5, 4.0]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, p in enumerate(p_values):
    corners_list, _, _ = find_corners(image1_gray, sg, p, k_harris, theta_corner)
    interest_points_visualization(image1_rgb, corners_list, ax=axes[col])
    axes[col].set_title(f'$\\rho$ = {p}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

Για διαφορετικές τιμές του $k$ έχουμε:

In [ ]:
sg = 2
p = 2.5
theta_corner = 0.005

k_values = [0.01, 0.05, 0.2]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, k in enumerate(k_values):
    corners_list, _, _ = find_corners(image1_gray, sg, p, k, theta_corner)
    interest_points_visualization(image1_rgb, corners_list, ax=axes[col])
    axes[col].set_title(f'$k$ = {k}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

Για διαφορετικές τιμές του $θ_{corn}$ έχουμε:

In [ ]:
sg = 2
p = 2.5
k_harris = 0.05

theta_values = [0.001, 0.005, 0.01]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, theta in enumerate(theta_values):
    corners_list, _, _ = find_corners(image1_gray, sg, p, k_harris, theta)
    interest_points_visualization(image1_rgb, corners_list, ax=axes[col])
    axes[col].set_title(f'$\\theta_{{corn}}$ = {theta}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

### 2.2

In [ ]:
# Function that finds corners in the image at multiple scales 
def find_multi_corners(image, sg0, p0, k, theta_corner, s_factor, N):
    
    # Initialize lists to store corners, LoG images, and scales for each level
    all_corners = []
    all_Log = []
    all_scales = []
    
    for i in range(N):
        sg = (s_factor ** i) * sg0
        p = (s_factor ** i) * p0
        
        # Find corners and LoG image for the current scale
        corners, _, _, log_image = find_corners(image, sg, p, k, theta_corner, return_log=True)
        
        all_corners.append(corners)
        all_Log.append(log_image)
        all_scales.append(sg)
    
    # Non-maximum suppression across scales
    final_corners = []
    for i in range(N): # Loop through each scale level
        for (x, y, _) in all_corners[i]: # Loop through each corner in the current scale level
            xi, yi = int(x), int(y)
            
            curr = all_Log[i][yi, xi] # LoG value at the current corner
            prev = all_Log[i-1][yi, xi] if i > 0 else -np.inf # LoG value at the previous scale level (or -inf if it's the first level)
            nxt = all_Log[i+1][yi, xi] if i < N-1 else -np.inf # LoG value at the next scale level (or -inf if it's the last level)
            
            if curr > prev and curr > nxt:
                final_corners.append([x, y, all_scales[i]]) # Append the corner with its corresponding scale if it's a local maximum across scales
    
    return np.array(final_corners)

Το αποτέλεσμα της στην εικόνα $\text{solar}$ είναι:

In [ ]:
sg0 = 2 # Initial standard deviation for the first Gaussian kernel
p0 = 2.5 # Initial standard deviation for the second Gaussian kernel
k = 0.05 # Harris detector free parameter
theta_corner = 0.005 # Threshold for corner detection
s_factor = 1.5 # Scale factor for increasing the scale
N = 4 # Number of scales

fig, ax = plt.subplots(figsize=(6, 6))  # μόνο ένα axes
name, (img_rgb, img_gray) = list(images.items())[0]  # παίρνουμε το πρώτο ζευγάρι εικόνας
multi_corners = find_multi_corners(img_gray, sg0, p0, k_harris, theta_corner, s_factor, N)
interest_points_visualization(img_rgb, multi_corners, ax=ax)

ax.set_title(name, fontsize=16, fontweight='medium', pad=15)
ax.axis('off')

plt.show()

Για διαφορετικές τιμές του $Ν$ έχουμε:

In [ ]:
sg0 = 2
p0 = 2.5
k_harris = 0.05
theta_corner = 0.005
s_factor = 1.5

N_values = [2, 4, 8]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, N in enumerate(N_values):
    multi_corners = find_multi_corners(image1_gray, sg0, p0, k_harris, theta_corner, s_factor, N)
    interest_points_visualization(image1_rgb, multi_corners, ax=axes[col])
    axes[col].set_title(f'N = {N}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

Για διαφορετικές τιμές του $s$ έχουμε:

In [ ]:
sg0 = 2
p0 = 2.5
k_harris = 0.05
theta_corner = 0.005
N = 4

s_values = [1.2, 1.5, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, s_factor in enumerate(s_values):
    multi_corners = find_multi_corners(image1_gray, sg0, p0, k_harris, theta_corner, s_factor, N)
    interest_points_visualization(image1_rgb, multi_corners, ax=axes[col])
    axes[col].set_title(f's = {s_factor}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

### 2.3

In [ ]:
# Function that finds blobs in the image using the determinant of the Hessian method
def find_blobs(image, s, theta_blob, return_R=False):

    image = image.astype(np.float64) / image.max() # Normalize to [0,1] for numerical stability and consistent thresholding

    Ns = 2 * np.ceil(3 * s) + 1
    Gs_1d = cv2.getGaussianKernel(int(Ns), s)
    Gs = Gs_1d @ Gs_1d.T

    Is = cv2.filter2D(image, -1, Gs)

    # Compute the second-order derivatives for the determinant of the Hessian
    grad_x = np.gradient(Is, axis=1)
    grad_y = np.gradient(Is, axis=0)
    Lxx = np.gradient(grad_x, axis=1)
    Lyy = np.gradient(grad_y, axis=0)
    Lxy = np.gradient(grad_y, axis=1)

    # Compute the determinant of the Hessian
    detH = Lxx * Lyy - Lxy ** 2

    B_sq = disk_strel(Ns)
    local_max = (detH == cv2.dilate(detH, B_sq))

    detHmax = np.max(detH)
    above_threshold = detH > theta_blob * detHmax

    blobs_mask = local_max & above_threshold
    y_blobs, x_blobs = np.where(blobs_mask)
    blobs_list = np.column_stack([x_blobs, y_blobs, s * np.ones(len(x_blobs))])

    if return_R:
        return blobs_list, detH
    return blobs_list

In [ ]:
s = 2
theta_blob = 0.005

fig, ax = plt.subplots(figsize=(6, 6))  
name, (img_rgb, img_gray) = list(images.items())[1] 
blobs_list = find_blobs(img_gray, s, theta_blob)
interest_points_visualization(img_rgb, blobs_list, ax=ax)

ax.set_title(name, fontsize=16, fontweight='medium', pad=15)
ax.axis('off')

plt.show()

Για διαφορετικές τιμές του $σ$ έχουμε:

In [ ]:
s_values = [0.5, 2, 4.0]
theta_blob = 0.005

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, s in enumerate(s_values):
    blobs_list = find_blobs(image2_gray, s, theta_blob)
    interest_points_visualization(image2_rgb, blobs_list, ax=axes[col])
    axes[col].set_title(f'$\\sigma$ = {s}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

plt.tight_layout()
plt.show()

Για διαφορετικές τιμές του $θ_{blob}$ έχουμε:

In [ ]:
s = 2
theta_values = [0.001, 0.005, 0.01]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, theta in enumerate(theta_values):
    blobs_list = find_blobs(image2_gray, s, theta)
    interest_points_visualization(image2_rgb, blobs_list, ax=axes[col])
    axes[col].set_title(f'$\\theta_{{blob}}$ = {theta}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

plt.tight_layout()
plt.show()

### 2.4

In [ ]:
# Function that finds blobs in the image at multiple scales using the determinant of the Hessian method
def find_multi_blobs(image, s0, theta_blob, s_factor, N):
    
    all_blobs = []
    all_detH = []  
    all_scales = []
    
    for i in range(N):
        s = (s_factor ** i) * s0
        
        # Find blobs and detH for the current scale
        blobs, detH = find_blobs(image, s, theta_blob, return_R=True)  
        
        all_blobs.append(blobs)
        all_detH.append(detH)  
        all_scales.append(s)
    
    # Non-maximum suppression across scales
    final_blobs = []
    for i in range(N):
        for (x, y, _) in all_blobs[i]:
            xi, yi = int(x), int(y)
            
            curr = all_detH[i][yi, xi]  
            prev = all_detH[i-1][yi, xi] if i > 0 else -np.inf  
            nxt = all_detH[i+1][yi, xi] if i < N-1 else -np.inf 
            
            if curr > prev and curr > nxt:
                final_blobs.append([x, y, all_scales[i]])
    
    return np.array(final_blobs)

In [ ]:
s0 = 2
theta_blob = 0.005
s_factor = 1.5
N = 4

fig, ax = plt.subplots(figsize=(6, 6))  
name, (img_rgb, img_gray) = list(images.items())[1]  
multi_blobs = find_multi_blobs(img_gray, s0, theta_blob, s_factor, N)
interest_points_visualization(img_rgb, multi_blobs, ax=ax)

ax.set_title(name, fontsize=16, fontweight='medium', pad=15)
ax.axis('off')

plt.show()

Για διαφορετικές τιμές του $Ν$ έχουμε:

In [ ]:
s0 = 2
theta_blob = 0.005
s_factor = 1.5

N_values = [2, 4, 8]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, N in enumerate(N_values):
    multi_blobs = find_multi_blobs(image2_gray, s0, theta_blob, s_factor, N)
    interest_points_visualization(image2_rgb, multi_blobs, ax=axes[col])
    axes[col].set_title(f'N = {N}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

plt.tight_layout()
plt.show()

Για διαφορετικές τιμές του $s$ έχουμε:

In [ ]:
s0 = 2
theta_blob = 0.005
N = 4

s_values = [1.2, 1.5, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(wspace=0.1)

for col, s_factor in enumerate(s_values):
    multi_blobs = find_multi_blobs(image2_gray, s0, theta_blob, s_factor, N)
    interest_points_visualization(image2_rgb, multi_blobs, ax=axes[col])
    axes[col].set_title(f's = {s_factor}', fontsize=16, fontweight='medium', pad=15)
    axes[col].axis('off')

plt.tight_layout()
plt.show()

## Μέρος 3

### 3.1

#### 3.1.1

In [ ]:
sg0 = 2
p0 = 2.5
k_harris = 0.05
theta_corner = 0.005
s_factor = 1.5
N = 4
s0 = 2
theta_blob = 0.005

# Combination 1: multi_corners + SURF
detect_fun = lambda I: find_multi_corners(I, sg0, p0, k, theta_corner, s_factor, N)
desc_fun = lambda I, kp: p3.featuresSURF(I, kp)
feats_corners_surf = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='corners_surf.pkl')

# Combination 2: multi_corners + HOG
detect_fun = lambda I: find_multi_corners(I, sg0, p0, k, theta_corner, s_factor, N)
desc_fun = lambda I, kp: p3.featuresHOG(I, kp)
feats_corners_hog = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='corners_hog.pkl')

# Combination 3: multi_blobs + SURF
detect_fun = lambda I: find_multi_blobs(I, s0, theta_blob, s_factor, N)
desc_fun = lambda I, kp: p3.featuresSURF(I, kp)
feats_blobs_surf = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='blobs_surf.pkl')

# Combination 4: multi_blobs + HOG
detect_fun = lambda I: find_multi_blobs(I, s0, theta_blob, s_factor, N)
desc_fun = lambda I, kp: p3.featuresHOG(I, kp)
feats_blobs_hog = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='blobs_hog.pkl')

#### 3.1.2-4

In [ ]:
feats_corners_surf = p3.FeatureExtraction(None, None, loadFile='corners_surf.pkl')
feats_corners_hog  = p3.FeatureExtraction(None, None, loadFile='corners_hog.pkl')
feats_blobs_surf   = p3.FeatureExtraction(None, None, loadFile='blobs_surf.pkl')
feats_blobs_hog    = p3.FeatureExtraction(None, None, loadFile='blobs_hog.pkl')

# Evaluate the combinations using 5-fold cross-validation
accs_corners_surf = []
for k in range(5):
    # Split into a training set and a test set
    data_train, label_train, data_test, label_test = p3.createTrainTest(feats_corners_surf, k)

    # Perform kmeans to find centroids for clusters
    BOF_tr, BOF_ts = p3.BagOfWords(data_train, data_test)

    # Train an svm on the training set and make predictions on the test set
    acc, preds, probas = p3.svm(BOF_tr, label_train, BOF_ts, label_test)
    accs_corners_surf.append(acc)

print('Mean accuracy for corners with SURF descriptors: {:.3f}%'.format(100.0*np.mean(accs_corners_surf)))

accs_corners_hog = []
for k in range(5):
    # Split into a training set and a test set
    data_train, label_train, data_test, label_test = p3.createTrainTest(feats_corners_hog, k)

    # Perform Kmeans to find centroids for clusters
    BOF_tr, BOF_ts = p3.BagOfWords(data_train, data_test)

    # Train an svm on the training set and make predictions on the test set
    acc, preds, probas = p3.svm(BOF_tr, label_train, BOF_ts, label_test)
    accs_corners_hog.append(acc)

print('Mean accuracy for corners with HOG descriptors: {:.3f}%'.format(100.0*np.mean(accs_corners_hog)))

accs_blobs_surf = []
for k in range(5):
    # Split into a training set and a test set
    data_train, label_train, data_test, label_test = p3.createTrainTest(feats_blobs_surf, k)

    # Perform Kmeans to find centroids for clusters
    BOF_tr, BOF_ts = p3.BagOfWords(data_train, data_test)

    # Train an svm on the training set and make predictions on the test set
    acc, preds, probas = p3.svm(BOF_tr, label_train, BOF_ts, label_test)
    accs_blobs_surf.append(acc)

print('Mean accuracy for blobs with SURF descriptors: {:.3f}%'.format(100.0*np.mean(accs_blobs_surf)))

accs_blobs_hog = []
for k in range(5):
    # Split into a training set and a test set
    data_train, label_train, data_test, label_test = p3.createTrainTest(feats_blobs_hog, k)

    # Perform Kmeans to find centroids for clusters
    BOF_tr, BOF_ts = p3.BagOfWords(data_train, data_test)

    # Train an svm on the training set and make predictions on the test set
    acc, preds, probas = p3.svm(BOF_tr, label_train, BOF_ts, label_test)
    accs_blobs_hog.append(acc)

print('Mean accuracy for blobs with HOG descriptors: {:.3f}%'.format(100.0*np.mean(accs_blobs_hog)))

### 3.2

#### 3.2.1.

In [ ]:
# Load the pre-trained MobileNetV3 Small model
weights = MobileNet_V3_Small_Weights.DEFAULT
model = torchvision.models.mobilenet_v3_small(weights=weights)

# List all the layers of the model
for name, module in model.named_modules():
    print(name, module.__class__.__name__)

# Get the preprocessing transformations for the model
preprocess = weights.transforms()
print(preprocess)

dataDir = 'Data'
categories = [
    ('person', 'TUGraz_person'),
    ('cars', 'TUGraz_cars'),
    ('bike', 'TUGraz_bike')
]

# Preprocess the data and store the tensors in a list
data_preprocessed = []

for label, (name, catDir) in enumerate(categories):
    img_list = sorted(os.listdir(os.path.join(dataDir, catDir)))
    class_tensors = []
    
    for img_file in img_list:
        if name not in img_file:
            continue
        
        img = Image.open(os.path.join(dataDir, catDir, img_file)).convert('RGB')
        
        img_tensor = preprocess(img)        # Apply the same preprocessing as used during training
        img_tensor = img_tensor.unsqueeze(0)
        
        class_tensors.append(img_tensor)
    
    data_preprocessed.append(class_tensors)

#### 3.2.2.

In [ ]:
from torchvision.models.feature_extraction import create_feature_extractor

# Evaluate the model in evaluation mode
model.eval()

# Create a feature extractor that keeps the output of the avgpool layer, which is the layer before the final classification layer
feature_extractor = create_feature_extractor(model, return_nodes={'avgpool': 'features'})

feats_mobilenet = []

for class_tensors in data_preprocessed:
    feats_class = []
    
    for img_tensor in class_tensors:
        with torch.no_grad():
            features = feature_extractor(img_tensor)
            
            feat_vector = features['features']  # shape: (1, 1280, 1, 1)
            feat_vector = feat_vector.squeeze().numpy()  # → (1280,)
            
            feats_class.append(feat_vector)
    
    feats_mobilenet.append(feats_class)

#### 3.2.3.

In [ ]:
# Evaluate the MobileNet features using 5-fold cross-validation
accs_mobilenet = []
for k in range(5):
    data_train, label_train, data_test, label_test = p3.createTrainTest(feats_mobilenet, k)
    
    # Convert lists to numpy arrays for SVM
    data_train = np.array(data_train)
    data_test = np.array(data_test)
    
    acc, preds, probas = p3.svm(data_train, label_train, data_test, label_test)
    accs_mobilenet.append(acc)

print('Mean accuracy for MobileNet: {:.3f}%'.format(100.0 * np.mean(accs_mobilenet)))

#### 3.2.4

In [ ]:
# Feature extractor for intermediate layers of MobileNetV3 Small
layer_extractor = create_feature_extractor(model, return_nodes={
    'features.1': 'layer1',
    'features.3': 'layer3',
    'features.6': 'layer6',
    'features.12': 'layer12'
})

# Sample images for visualization (one from each category)
sample_images = {
    'person': 'TUGraz_person/person_055.png',
    'car':    'TUGraz_cars/carsgraz_372.png',
    'bike':   'TUGraz_bike/bike_155.png'
}

for category, img_path in sample_images.items():
    img = Image.open(os.path.join(dataDir, img_path)).convert('RGB')
    img_tensor = preprocess(img).unsqueeze(0)

    with torch.no_grad():
        layer_outputs = layer_extractor(img_tensor)

    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    fig.suptitle(f'Feature maps - {category}', fontsize=28, fontweight='medium', y=1.02)

    # Display the original image in the first column
    for row in range(4):
        axes[row, 0].imshow(img)
        axes[row, 0].axis('off')
        if row == 0:
            axes[row, 0].set_title('Αρχική εικόνα', fontsize=18, fontweight='medium', pad=10)

    layer_labels = ['Layer 1', 'Layer 3', 'Layer 6', 'Layer 12']

    for row, (layer_name, output) in enumerate(layer_outputs.items()):
        feat_maps = output.squeeze().numpy()  # (C, H, W)
        
        # Normalize for better visualization
        for col in range(3):
            fmap = feat_maps[col]
            fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
            
            axes[row, col+1].imshow(fmap, cmap='viridis')
            axes[row, col+1].axis('off')
            axes[row, col+1].set_title(f'{layer_labels[row]} - ch{col+1}', fontsize=18, fontweight='medium', pad=10)

    plt.tight_layout()
    plt.show()
    

#### 3.2.5

Για το συγκεκριμένο ερώτημα χρησιμοποιηθήκε η μεταβλητή $\text{distort}$ η οποία βρισκόταν εντός της συνάρτησης $\text{FeatureExtraction}$. Στο τέλος του αρχείου δίνονται που κάναμε στη δοσμένη συνάρτηση.

3.1 Με θόρυβο

In [ ]:
sg0 = 2
p0 = 2.5
k_harris = 0.05
theta_corner = 0.005
s_factor = 1.5
N = 4
s0 = 2
theta_blob = 0.005

# Combination 1: multi_corners + SURF
detect_fun = lambda I: find_multi_corners(I, sg0, p0, k_harris, theta_corner, s_factor, N)
desc_fun = lambda I, kp: p3.featuresSURF(I, kp)
feats_corners_surf_noise = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='corners_surf_noise.pkl', distort=1)

# Combination 2: multi_corners + HOG
detect_fun = lambda I: find_multi_corners(I, sg0, p0, k_harris, theta_corner, s_factor, N)
desc_fun = lambda I, kp: p3.featuresHOG(I, kp)
feats_corners_hog_noise = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='corners_hog_noise.pkl', distort=1)

# Combination 3: multi_blobs + SURF
detect_fun = lambda I: find_multi_blobs(I, s0, theta_blob, s_factor, N)
desc_fun = lambda I, kp: p3.featuresSURF(I, kp)
feats_blobs_surf_noise = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='blobs_surf_noise.pkl', distort=1)

# Combination 4: multi_blobs + HOG
detect_fun = lambda I: find_multi_blobs(I, s0, theta_blob, s_factor, N)
desc_fun = lambda I, kp: p3.featuresHOG(I, kp)
feats_blobs_hog_noise = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='blobs_hog_noise.pkl', distort=1)

In [ ]:
feats_corners_surf_noise = p3.FeatureExtraction(None, None, loadFile='corners_surf_noise.pkl')
feats_corners_hog_noise  = p3.FeatureExtraction(None, None, loadFile='corners_hog_noise.pkl')
feats_blobs_surf_noise   = p3.FeatureExtraction(None, None, loadFile='blobs_surf_noise.pkl')
feats_blobs_hog_noise    = p3.FeatureExtraction(None, None, loadFile='blobs_hog_noise.pkl')

# Evaluate
combinations = {
    'corners + SURF': feats_corners_surf_noise,
    'corners + HOG':  feats_corners_hog_noise,
    'blobs + SURF':   feats_blobs_surf_noise,
    'blobs + HOG':    feats_blobs_hog_noise,
}

for name, feats in combinations.items():
    accs = []
    for k in range(5):
        data_train, label_train, data_test, label_test = p3.createTrainTest(feats, k)
        BOF_tr, BOF_ts = p3.BagOfWords(data_train, data_test)
        acc, preds, probas = p3.svm(BOF_tr, label_train, BOF_ts, label_test)
        accs.append(acc)
    print('Mean accuracy for {} with noise: {:.3f}%'.format(name, 100.0*np.mean(accs)))

3.1 Με στροφή

In [ ]:
# Combination 1: multi_corners + SURF
detect_fun = lambda I: find_multi_corners(I, sg0, p0, k_harris, theta_corner, s_factor, N)
desc_fun = lambda I, kp: p3.featuresSURF(I, kp)
feats_corners_surf_rot = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='corners_surf_rot.pkl', distort=2)

# Combination 2: multi_corners + HOG
detect_fun = lambda I: find_multi_corners(I, sg0, p0, k_harris, theta_corner, s_factor, N)
desc_fun = lambda I, kp: p3.featuresHOG(I, kp)
feats_corners_hog_rot = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='corners_hog_rot.pkl', distort=2)

# Combination 3: multi_blobs + SURF
detect_fun = lambda I: find_multi_blobs(I, s0, theta_blob, s_factor, N)
desc_fun = lambda I, kp: p3.featuresSURF(I, kp)
feats_blobs_surf_rot = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='blobs_surf_rot.pkl', distort=2)

# Combination 4: multi_blobs + HOG
detect_fun = lambda I: find_multi_blobs(I, s0, theta_blob, s_factor, N)
desc_fun = lambda I, kp: p3.featuresHOG(I, kp)
feats_blobs_hog_rot = p3.FeatureExtraction(detect_fun, desc_fun, saveFile='blobs_hog_rot.pkl', distort=2)

In [ ]:
feats_corners_surf_rot = p3.FeatureExtraction(None, None, loadFile='corners_surf_rot.pkl')
feats_corners_hog_rot  = p3.FeatureExtraction(None, None, loadFile='corners_hog_rot.pkl')
feats_blobs_surf_rot   = p3.FeatureExtraction(None, None, loadFile='blobs_surf_rot.pkl')
feats_blobs_hog_rot    = p3.FeatureExtraction(None, None, loadFile='blobs_hog_rot.pkl')

# Evaluate
combinations = {
    'corners + SURF': feats_corners_surf_rot,
    'corners + HOG':  feats_corners_hog_rot,
    'blobs + SURF':   feats_blobs_surf_rot,
    'blobs + HOG':    feats_blobs_hog_rot,
}

for name, feats in combinations.items():
    accs = []
    for k in range(5):
        data_train, label_train, data_test, label_test = p3.createTrainTest(feats, k)
        BOF_tr, BOF_ts = p3.BagOfWords(data_train, data_test)
        acc, preds, probas = p3.svm(BOF_tr, label_train, BOF_ts, label_test)
        accs.append(acc)
    print('Mean accuracy for {} with rotation: {:.3f}%'.format(name, 100.0*np.mean(accs)))

3.2

In [ ]:
# Distortion functions for noise and rotation
def add_noise_pil(img_pil):
    img_np = np.array(img_pil).astype(np.float64)
    noise = np.random.normal(0, 35, img_np.shape)
    noisy = np.clip(img_np + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy)

def add_rotation_pil(img_pil):
    angle = np.random.uniform(-45, 45)
    return img_pil.rotate(angle)

def extract_mobilenet_features(distort_fn=None):
    feats = []
    for label, (name, catDir) in enumerate(categories):
        img_list = sorted(os.listdir(os.path.join(dataDir, catDir)))
        feats_class = []
        for img_file in img_list:
            if name not in img_file:
                continue
            img = Image.open(os.path.join(dataDir, catDir, img_file)).convert('RGB')
            
            if distort_fn is not None:
                img = distort_fn(img)
            
            img_tensor = preprocess(img).unsqueeze(0)
            with torch.no_grad():
                features = feature_extractor(img_tensor)
                feat_vector = features['features'].squeeze().numpy()
            feats_class.append(feat_vector)
        feats.append(feats_class)
    return feats

3.2 θόρυβος

In [ ]:
# Extract features with noise distortion
feats_mobilenet_noise = extract_mobilenet_features(distort_fn=add_noise_pil)

accs_mobilenet_noise = []
for k in range(5):
    data_train, label_train, data_test, label_test = p3.createTrainTest(feats_mobilenet_noise, k)
    data_train = np.array(data_train)
    data_test = np.array(data_test)
    acc, preds, probas = p3.svm(data_train, label_train, data_test, label_test)
    accs_mobilenet_noise.append(acc)
print('Mean accuracy for MobileNet with noise: {:.3f}%'.format(100.0 * np.mean(accs_mobilenet_noise)))

In [ ]:
# Extract features with noise distortion
layer_extractor = create_feature_extractor(model, return_nodes={
    'features.1': 'layer1',
    'features.3': 'layer3',
    'features.6': 'layer6',
    'features.12': 'layer12'
})

sample_images_paths = {
    'person': 'TUGraz_person/person_055.png',
    'car':    'TUGraz_cars/carsgraz_372.png',
    'bike':   'TUGraz_bike/bike_155.png'
}

# Extract features for the sample images with noise distortion and visualize feature maps
sample_images = {}

for category, img_path in sample_images_paths.items():
    img = Image.open(os.path.join(dataDir, img_path)).convert('RGB')
    noisy_img = add_noise_pil(img)
    img_tensor = preprocess(noisy_img).unsqueeze(0)

    with torch.no_grad():
        layer_outputs = layer_extractor(img_tensor)

    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    fig.suptitle(f'Feature maps - {category}', fontsize=28, fontweight='medium', y=1.02)

    # Display the noisy image in the first column
    for row in range(4):
        axes[row, 0].imshow(noisy_img)
        axes[row, 0].axis('off')
        if row == 0:
            axes[row, 0].set_title('Αρχική εικόνα', fontsize=18, fontweight='medium', pad=10)

    layer_labels = ['Layer 1', 'Layer 3', 'Layer 6', 'Layer 12']

    for row, (layer_name, output) in enumerate(layer_outputs.items()):
        feat_maps = output.squeeze().numpy()  # (C, H, W)
        
        # Normalize for better visualization
        for col in range(3):
            fmap = feat_maps[col]
            fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
            
            axes[row, col+1].imshow(fmap, cmap='viridis')
            axes[row, col+1].axis('off')
            axes[row, col+1].set_title(f'{layer_labels[row]} - ch{col+1}', fontsize=18, fontweight='medium', pad=10)

    plt.tight_layout()
    plt.show()
    

3.2 Περιστροφή

In [ ]:
# Extract features with rotation distortion
feats_mobilenet_rot = extract_mobilenet_features(distort_fn=add_rotation_pil)

accs_mobilenet_rot = []
for k in range(5):
    data_train, label_train, data_test, label_test = p3.createTrainTest(feats_mobilenet_rot, k)
    data_train = np.array(data_train)
    data_test = np.array(data_test)
    acc, preds, probas = p3.svm(data_train, label_train, data_test, label_test)
    accs_mobilenet_rot.append(acc)
print('Mean accuracy for MobileNet with rotation: {:.3f}%'.format(100.0 * np.mean(accs_mobilenet_rot)))

In [ ]:
# Feature extractor for intermediate layers of MobileNetV3 Small with rotation distortion
layer_extractor = create_feature_extractor(model, return_nodes={
    'features.1': 'layer1',
    'features.3': 'layer3',
    'features.6': 'layer6',
    'features.12': 'layer12'
})

sample_images_paths = {
    'person': 'TUGraz_person/person_055.png',
    'car':    'TUGraz_cars/carsgraz_372.png',
    'bike':   'TUGraz_bike/bike_155.png'
}

# Extract features for the sample images with rotation distortion and visualize feature maps
sample_images = {}

for category, img_path in sample_images_paths.items():
    img = Image.open(os.path.join(dataDir, img_path)).convert('RGB')
    rot_img = add_rotation_pil(img)
    img_tensor = preprocess(rot_img).unsqueeze(0)

    with torch.no_grad():
        layer_outputs = layer_extractor(img_tensor)

    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    fig.suptitle(f'Feature maps - {category}', fontsize=28, fontweight='medium', y=1.02)

    # Display the rotated image in the first column
    for row in range(4):
        axes[row, 0].imshow(rot_img)
        axes[row, 0].axis('off')
        if row == 0:
            axes[row, 0].set_title('Αρχική εικόνα', fontsize=18, fontweight='medium', pad=10)

    layer_labels = ['Layer 1', 'Layer 3', 'Layer 6', 'Layer 12']

    for row, (layer_name, output) in enumerate(layer_outputs.items()):
        feat_maps = output.squeeze().numpy()  # (C, H, W)
        
        # Normalize for better visualization
        for col in range(3):
            fmap = feat_maps[col]
            fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
            
            axes[row, col+1].imshow(fmap, cmap='viridis')
            axes[row, col+1].axis('off')
            axes[row, col+1].set_title(f'{layer_labels[row]} - ch{col+1}', fontsize=18, fontweight='medium', pad=10)

    plt.tight_layout()
    plt.savefig(f'feature_maps_{category}.png', dpi=150, bbox_inches='tight')
    plt.show()
    

Αλλαγμένος κώδικας του $\text{cv26\_lab1\_part3\_utils.py}$:

In [ ]:
# Noise and rotation functions for OpenCV images
def add_noise(image):
    noise = np.random.normal(0, 35, image.shape).astype(np.float64)
    noisy = np.clip(image.astype(np.float64) + noise, 0, 255).astype(np.uint8)
    return noisy

def add_rotation(image):
    angle = np.random.uniform(-45, 45)
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    return rotated

def distort_image(image, distort_type):
    if distort_type == 1:
        return add_noise(image)
    elif distort_type == 2:
        return add_rotation(image)
    return image

def FeatureExtraction(detector_fun, descriptor_fun, loadFile=None, saveFile=None, distort=0):
    ''' Extract features using the descriptor provided in constructor'''

    if loadFile is not None:
        X_full = pickle.load(open(loadFile,'rb'))
        return X_full

    # The following should correspond to the data
    dataDir = 'Data'
    categories = [
        ('person','TUGraz_person'),
        ('cars','TUGraz_cars'),
        ('bike','TUGraz_bike')
    ]

    num_cpus = os.cpu_count()
    if num_cpus is None:
        num_procs = 1
    else:
        num_procs = min(3,num_cpus-1)

    start_time = time.time()
    im_full = []
    for name, catDir in categories:
        img_list = sorted(os.listdir(os.path.join(dataDir, catDir)))
        im_class = []
        count = 0
        for img_file in tqdm(img_list, total=len(img_list), desc=f"Reading {name} images"):
            if name not in img_file:
                continue
            I = cv2.cvtColor(cv2.imread(os.path.join(dataDir, catDir, img_file)), cv2.COLOR_BGR2GRAY)
            if distort:
                I = distort_image(I, distort)
            I = cv2.resize(I, (0,0), fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
            im_class.append(I)
        # For visualization of distortions, save side-by-side comparisons for the first image of each category
        if len(img_list) > 0:
            # Only visualize the first image for each category
            for img_file in img_list:
                if name in img_file:
                    path = os.path.join(dataDir, catDir, img_file)
                    original = cv2.imread(path)  # BGR

                    gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
                    if distort:
                        distorted_gray = distort_image(gray, distort)
                    else:
                        distorted_gray = gray

                    gray = cv2.resize(gray, (0,0), fx=0.5, fy=0.5)
                    distorted_gray = cv2.resize(distorted_gray, (0,0), fx=0.5, fy=0.5)

                    # plot side-by-side
                    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

                    axes[0].imshow(gray, cmap='gray')
                    axes[0].set_title("Original")
                    axes[0].axis('off')

                    axes[1].imshow(distorted_gray, cmap='gray')
                    axes[1].set_title("Distorted")
                    axes[1].axis('off')

                    plt.tight_layout()
                    plt.savefig(f'comparison_noise_{name}.png', dpi=150, bbox_inches='tight')
                    plt.close()

                    break 
        im_full.append((detector_fun,descriptor_fun,im_class))


    start_time = time.time()

    X_full = list(map(FeatureExtractionThread, im_full))

    print('Time for feature extraction: {:.3f}'.format(time.time()-start_time))

    if saveFile is not None:
        pickle.dump(X_full, open(saveFile,'wb'))

    return X_full